# 03 - Train Supervised Baselines

This notebook is a Colab runner for the four supervised baseline experiments:

- `resnet18_none`
- `resnet18_imagenet`
- `vit_s16_none`
- `vit_s16_imagenet`

Training logic stays in the Python scripts. This notebook only prepares Colab and calls those scripts.

## 1. Mount Google Drive

Mount Drive so results can be copied to persistent storage after training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Pull The Repository

Use the current repository as the source of scripts, configs, data manifests, and baseline training code.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    print(f'Repository exists at {REPO_ROOT}. Pulling latest changes...')
    %cd {REPO_ROOT}
    !git pull
else:
    print(f'Cloning repository to {REPO_ROOT}...')
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

print('Current working directory:', Path.cwd())

## 3. Install Dependencies

Install project dependencies when `requirements.txt` exists, while keeping Colab's built-in PyTorch/CUDA stack. The repository requirements may contain CUDA-specific package pins that do not resolve through Colab's default pip index.

In [ ]:
import sys
import subprocess

requirements_path = REPO_ROOT / 'requirements.txt'
filtered_requirements_path = REPO_ROOT / 'requirements_colab_filtered.txt'

SKIP_PREFIXES = (
    'torch',
    'torchvision',
    'torchaudio',
    'triton',
    'nvidia-',
)

if requirements_path.exists():
    filtered_lines = []
    skipped_lines = []
    for raw_line in requirements_path.read_text().splitlines():
        line = raw_line.strip()
        package_name = line.split('==')[0].split('>=')[0].split('<=')[0].split('~=')[0].lower()
        if not line or line.startswith('#'):
            filtered_lines.append(raw_line)
        elif package_name.startswith(SKIP_PREFIXES):
            skipped_lines.append(raw_line)
        else:
            filtered_lines.append(raw_line)

    filtered_requirements_path.write_text('\n'.join(filtered_lines) + '\n')
    print('Skipped Colab runtime packages:', skipped_lines)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(filtered_requirements_path)])
else:
    print('WARNING: requirements.txt not found. Skipping dependency installation.')

import torch
print('Torch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())

## 4. Editable Runner Variables

Set the run flags to control which experiments execute. `OUTPUT_ROOT` is used only by the optional Drive copy cells; the scripts write to `results/experiments/<experiment_id>/` from their configs.

In [ ]:
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')
OUTPUT_ROOT = Path('/content/drive/MyDrive/contrastive-synthesis-medcls_CVProject/results/experiments')

RUN_RESNET_NONE = True
RUN_RESNET_IMAGENET = True
RUN_VIT_NONE = True
RUN_VIT_IMAGENET = True

%cd {REPO_ROOT}
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 5. Lightweight Checks

Run the input checker and compile the training scripts before launching any experiment.

In [ ]:
!python scripts/check_experiment_inputs.py
!python -m py_compile scripts/run_classification_resnet.py scripts/run_classification_vit.py

## 6. Experiment: resnet18_none

Train ResNet18 from random initialization on the fixed real labeled train/val/test manifests.

In [ ]:
if RUN_RESNET_NONE:
    !python scripts/run_classification_resnet.py \
      --config configs/experiments/resnet18/none.yaml \
      --manifest-dir data/manifests
else:
    print('Skipping resnet18_none')

## 7. Experiment: resnet18_imagenet

Fine-tune ImageNet-pretrained ResNet18 on the fixed real labeled train/val/test manifests.

In [ ]:
if RUN_RESNET_IMAGENET:
    !python scripts/run_classification_resnet.py \
      --config configs/experiments/resnet18/imagenet.yaml \
      --manifest-dir data/manifests
else:
    print('Skipping resnet18_imagenet')

## 8. Experiment: vit_s16_none

Train ViT-S/16 from random initialization on the fixed real labeled train/val/test manifests.

In [ ]:
if RUN_VIT_NONE:
    !python scripts/run_classification_vit.py \
      --config configs/experiments/vit_s16/none.yaml \
      --manifest-dir data/manifests
else:
    print('Skipping vit_s16_none')

## 9. Experiment: vit_s16_imagenet

Fine-tune ImageNet-pretrained ViT-S/16 on the fixed real labeled train/val/test manifests.

In [ ]:
if RUN_VIT_IMAGENET:
    !python scripts/run_classification_vit.py \
      --config configs/experiments/vit_s16/imagenet.yaml \
      --manifest-dir data/manifests
else:
    print('Skipping vit_s16_imagenet')

## 10. Optional: Copy Results To Google Drive

Use this after training to persist `results/experiments/` in Drive.

In [ ]:
COPY_RESULTS_TO_DRIVE = False

if COPY_RESULTS_TO_DRIVE:
    OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    !mkdir -p "{OUTPUT_ROOT.parent}"
    !rsync -av results/experiments/ "{OUTPUT_ROOT}/"
else:
    print('Set COPY_RESULTS_TO_DRIVE = True to copy results to Drive.')

## 11. Inspect Expected Result Files

After each run, the experiment folder should contain `config_resolved.yaml`, `best_checkpoint.pth`, `metrics.json`, `classification_report.csv`, and `confusion_matrix.png`.

In [ ]:
!find results/experiments -maxdepth 2 -type f | sort || true